In [ ]:
# IMPORTING LIBRARIES
# ============================
import os
import re
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import pipeline, AutoTokenizer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    AutoModelForSeq2SeqLM
)

Reading DATASET (Example: a CSV file)

In [ ]:
# Load your data here.
# This should be a pandas DataFrame with at least a 'text' column for input.
# For classification tasks, you'll also need a 'label' column.

# Example: Load from a CSV
# my_dataset = pd.read_csv('/path_to_your_file.csv')

# Example: Create a dummy DataFrame for demonstration
my_dataset = pd.DataFrame({
    'text': [
        "Patient has severe pneumonia, elevated white blood cell count, and fever. Admitted to ICU.",
        "Routine check-up, no significant findings. Patient healthy.",
        "History of hypertension and diabetes. Scheduled for follow-up on medication review.",
        "Fracture of the tibia, requiring surgical intervention. Post-op care initiated.",
        "Generalized anxiety disorder, mild to moderate. Recommend cognitive behavioral therapy."
    ],
    'label': ["pneumonia", "healthy", "chronic_disease", "fracture", "anxiety"]
})

print(f"Loaded dataset shape: {my_dataset.shape}")
print("First 5 rows of the dataset:")
display(my_dataset.head())

Loaded dataset shape: (5, 2)
First 5 rows of the dataset:


,text,label
0,"Patient has severe pneumonia, elevated white b...",pneumonia
1,"Routine check-up, no significant findings. Pat...",healthy
2,History of hypertension and diabetes. Schedule...,chronic_disease
3,"Fracture of the tibia, requiring surgical inte...",fracture
4,"Generalized anxiety disorder, mild to moderate...",anxiety


## What are User Access Tokens?

https://huggingface.co/docs/hub/en/security-tokens


In [ ]:
from huggingface_hub import login

# Replace "YOUR_ACCESS_TOKEN" with your actual token
login(token="YOUR_ACCESS_TOKEN")

## Model selections and Configs

In [ ]:

# 1. Optional: Specific GPU Config (e.g., for OSCER, remove if not applicable)
# os.environ['HF_HOME'] = '/scratch/reza/huggingface_home'
# os.environ['TRANSFORMERS_CACHE'] = '/scratch/reza/huggingface_cache'
# os.environ['HF_DATASETS_CACHE'] = '/scratch/reza/huggingface_datasets'
# os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 2. LLM Model Mapping (Generative Models for Text Generation/Classification via Prompting)
# Note: Ensure you have granted access on HuggingFace for Llama/Mistral models if used
gen_models = {
    "Llama-3.2-3B": "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen-3B" : "Qwen/Qwen2.5-3B-Instruct",
    "Mistral-7B": "mistralai/Mistral-7B-Instruct-v0.3",
    "Phi-4": "microsoft/phi-4"
}

# 3. Summarization Models (Seq2Seq models)
summarization_models = {
    "Medical Summarization": "Falconsai/medical_summarization",
    "BART Summarizer": "KipperDev/bart_summarizer_model",
    "T5-base": "t5-base",
    "Pegasus-large": "google/pegasus-large"
}

# 4. Classification Models (Encoder-only for Sequence Classification)
classification_models = {
    "PubMedBERT": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    "BERT-base-uncased": "bert-base-uncased",
    "RoBERTa-base": "roberta-base"
}

print("Model selection dictionaries defined. Choose your model for each task.")

Model selection dictionaries defined. Choose your model for each task.


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# --- LLM for Summarization ---
# This section demonstrates how to use a pre-trained LLM for text summarization.
# You can choose from various models available on Hugging Face using the 'summarization_models' dictionary defined above.

# Select a summarization model
model_choice = "Medical Summarization" # @param ["Medical Summarization", "BART Summarizer", "T5-base", "Pegasus-large"]
model_name = summarization_models[model_choice]

print(f"Loading summarization model: {model_name}")

tokenizer_summarization = AutoTokenizer.from_pretrained(model_name)
model_summarization = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Initialize the summarization functionality
# Due to recurring issues with the 'summarization' and 'text2text-generation'
# pipeline tasks not being recognized, we will directly use the model's generate method.
# This approach provides explicit control and works around pipeline compatibility problems.

# Set device and move model to it
device_idx = 0 if torch.cuda.is_available() else -1
if device_idx != -1:
    model_summarization.to(f'cuda:{device_idx}')

def summarizer_pipeline(texts, tokenizer, model, generation_kwargs):
    """
    Custom summarizer function using model.generate() directly.
    Takes a list of texts and returns a list of dictionaries with 'summary_text'.
    """
    summaries = []
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            max_length=tokenizer.model_max_length, # Ensure input is not too long
            truncation=True
        ).to(model.device) # Move inputs to the same device as model

        output_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            **generation_kwargs # Unpack generation arguments (max_length, min_length, do_sample)
        )
        # Decode the generated output and extract the summary
        decoded_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        summaries.append({'summary_text': decoded_output})
    return summaries

# Reassign summarizer_pipeline to our custom function wrapper
# This keeps the calling convention for subsequent cells consistent.
summarizer_pipeline = lambda texts: custom_summarizer_function(
    texts,
    tokenizer_summarization,
    model_summarization,
    {"max_length": 150, "min_length": 30, "do_sample": False} # Using the desired generation kwargs
)

print("Summarization functionality initialized using model.generate() directly.")

In [ ]:
# # --- Demo Basic Summarization ---
# # We will use a sample of the generic data from 'my_dataset' for demonstration.

# # Ensure your DataFrame has a 'text' column with the content to summarize.
texts_to_summarize = my_dataset['text'].head(5).tolist()

print("Generating summaries for the first 5 entries...")

# # Call the summarization pipeline. max_length, min_length, do_sample are set in generation_kwargs
# # during pipeline initialization, so they are not needed here.
summaries_output = summarizer_pipeline(texts_to_summarize)

# # Extract just the summary texts
generated_summary_texts = [s['summary_text'] for s in summaries_output]

print("Summaries generated.")

Generating summaries for the first 5 entries...
Summaries generated.


In [ ]:
for i, summary_text in enumerate(generated_summary_texts):
    print(f"Original Text {i+1}:\n{texts_to_summarize[i]}\n")
    print(f"Generated Summary {i+1}:\n{summary_text}\n")
    print("-" * 50 + "\n")

Original Text 1:
Patient has severe pneumonia, elevated white blood cell count, and fever. Admitted to ICU.

Generated Summary 1:
ICU severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated white SDRAM severe pneumonia elevated whit

In [ ]:
# --- Classification Model Setup ---
# This section demonstrates how to use a pre-trained model for text classification.
# You can choose from various models available on Hugging Face using the 'classification_models' dictionary defined above.

# Select a classification model
classification_model_choice = "PubMedBERT" # @param ["PubMedBERT", "BERT-base-uncased", "RoBERTa-base"]
classification_model_name = classification_models[classification_model_choice]

print(f"Loading classification model: {classification_model_name}")

tokenizer_classification = AutoTokenizer.from_pretrained(classification_model_name)
model_classification = AutoModelForSequenceClassification.from_pretrained(classification_model_name, num_labels=len(my_dataset['label'].unique()))

# Move model to GPU if available
if torch.cuda.is_available():
    model_classification.to('cuda')

print("Classification model and tokenizer loaded.")

In [ ]:
# --- Data Preparation for Classification ---

# Create a mapping from label names to IDs and vice-versa
label_to_id = {label: i for i, label in enumerate(my_dataset['label'].unique())}
id_to_label = {i: label for label, i in label_to_id.items()}

# Add an 'labels' column with numerical IDs to the dataset
my_dataset['labels'] = my_dataset['label'].map(label_to_id)

# Convert the pandas DataFrame to a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(my_dataset[['text', 'labels']])

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer_classification(examples['text'], truncation=True, padding='max_length')

# Apply tokenization to the dataset
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

print("Dataset prepared for classification:")
print(tokenized_dataset)
print("Label to ID mapping:", label_to_id)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset prepared for classification:
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5
})
Label to ID mapping: {'pneumonia': 0, 'healthy': 1, 'chronic_disease': 2, 'fracture': 3, 'anxiety': 4}


In [ ]:
# --- Classification Inference ---

# Create a DataLoader for inference (optional, but good practice for larger datasets)
from torch.utils.data import DataLoader
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_classification)

# Prepare the dataset for PyTorch
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

inference_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=2, # Use a small batch size for demonstration
    collate_fn=data_collator
)

# Perform inference
predictions = []
model_classification.eval() # Set model to evaluation mode

with torch.no_grad(): # Disable gradient calculations for inference
    for batch in inference_dataloader:
        # Move batch to the same device as the model
        batch = {k: v.to(model_classification.device) for k, v in batch.items()}
        outputs = model_classification(**batch)
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        predictions.extend(batch_predictions)

# Map predicted IDs back to original labels
predicted_labels = [id_to_label[p_id] for p_id in predictions]

# Display results
my_dataset['predicted_label'] = predicted_labels
print("Classification Results:")
display(my_dataset[['text', 'label', 'predicted_label']])

Classification Results:


,text,label,predicted_label
0,"Patient has severe pneumonia, elevated white b...",pneumonia,chronic_disease
1,"Routine check-up, no significant findings. Pat...",healthy,fracture
2,History of hypertension and diabetes. Schedule...,chronic_disease,fracture
3,"Fracture of the tibia, requiring surgical inte...",fracture,fracture
4,"Generalized anxiety disorder, mild to moderate...",anxiety,fracture
